In [ ]:
import torch
from tuned_lens.nn.lenses import TunedLens
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'meta-llama/Llama-2-7b-hf'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForCausalLM.from_pretrained(model_name)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tuned_lens = TunedLens.from_model_and_pretrained(model, map_location=device)
tuned_lens = tuned_lens.to(device)

input_ids_ring = tokenizer.encode(
    "The atomic number of B is "
)



targets_ring = input_ids_ring[1:] + [tokenizer.eos_token_id]


print(tokenizer.convert_ids_to_tokens(input_ids_ring))


from tuned_lens.plotting import PredictionTrajectory



predictition_traj_ring = PredictionTrajectory.from_lens_and_model(
    tuned_lens,
    model,
    tokenizer=tokenizer,
    input_ids=input_ids_ring,
    targets=targets_ring,
)



In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"

from plotly.subplots import make_subplots

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=("Entropy", "Forward KL", "Cross Entropy", "Max Probability"),
)

fig.add_trace(
    predictition_traj_ring.entropy().heatmap(
        colorbar_y=0.89, colorbar_len=0.25, textfont={'size':10}
    ),
    row=1, col=1
)

fig.add_trace(
    predictition_traj_ring.forward_kl().heatmap(
        colorbar_y=0.63, colorbar_len=0.25, textfont={'size':10}
    ),
    row=2, col=1
)

fig.add_trace(
    predictition_traj_ring.cross_entropy().heatmap(
        colorbar_y=0.37, colorbar_len=0.25, textfont={'size':10}
    ),
    row=3, col=1
)

fig.add_trace(
    predictition_traj_ring.max_probability().heatmap(
        colorbar_y=0.11, colorbar_len=0.25, textfont={'size':10}
    ),
    row=4, col=1
)

fig.update_layout(height=1600, width=1000, title_text="Tolkien's Tokens on visualized with the Tuned Lens")
fig.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
from tuned_lens.nn.lenses import TunedLens
from tuned_lens.plotting import PredictionTrajectory
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'meta-llama/Llama-2-7b-hf'
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tuned_lens = TunedLens.from_model_and_pretrained(model, map_location=device).to(device)

input_text = "In the periodic table, the atomic number of B is"
input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)

max_new_tokens = 10
with torch.no_grad():
    generated_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

# Split out the newly generated part
new_token_ids = generated_ids[0, input_ids.size(1):]
full_ids = generated_ids[0]

# Construct targets
targets = torch.cat([
    full_ids[1:], 
    torch.tensor([tokenizer.eos_token_id], device=device)
])

# Compute logits using Tuned Lens
prediction_traj = PredictionTrajectory.from_lens_and_model(
    tuned_lens,
    model,
    tokenizer=tokenizer,
    input_ids=full_ids,
    targets=targets
)

# logits.shape = (num_layers, seq_len, vocab_size)
probs = prediction_traj.log_probs


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Basic settings
num_layers = probs.shape[0]
seq_len = probs.shape[1]

start_idx = input_ids.size(1)
end_idx = full_ids.size(0)

layers = np.arange(num_layers)

# Set color palette
cmap = plt.get_cmap('tab10')  # Can be replaced with 'Set2', 'Paired', 'Dark2'
max_colors = cmap.N

# Limit the maximum number of tokens to display
MAX_TOKENS = 10

plt.figure(figsize=(6, 4))

token_count = 0

for pos in range(start_idx, end_idx - 1):
    token_id = full_ids[pos + 1].item()
    token_str = tokenizer.decode([token_id], clean_up_tokenization_spaces=False)

    token_log_probs = probs[:, pos, token_id]
    token_probs = np.exp(token_log_probs)

    color = cmap(token_count % max_colors)

    plt.plot(
        layers,
        token_probs,
        label=f"'{token_str}'",
        linewidth=1.5,
        color=color
    )

    # Mark the token as a red cross if it is in the top 5 for each layer
    for layer_i in range(num_layers):
        layer_log_probs = probs[layer_i, pos, :]
        layer_probs = np.exp(layer_log_probs)
        top5_ids = layer_probs.argsort()[-1:]

        if token_id in top5_ids:
            plt.scatter(
                layer_i,
                token_probs[layer_i],
                marker='o',
                s=30,
                color=color,
                zorder=1
            )

    token_count += 1
    if token_count >= MAX_TOKENS:
        break  # Control the number of tokens displayed to prevent clutter

# Beautification
plt.xlabel("Layer Index", fontsize=12)
plt.ylabel("Probability", fontsize=12)
# plt.title("Probability Trajectories for Generated Tokens (Tuned Lens)", fontsize=14)

plt.grid(True, linestyle='--', alpha=0.4)

# Place legend outside the plot to avoid overlap
plt.legend(
    loc='upper left',
    fontsize=8,     

)

plt.tight_layout()

plt.savefig("Results/token_prob_trajectories.png", dpi=300, bbox_inches='tight')


plt.show()  # Display the image (optional)